# Analise de Vendas em E-commerce no Brasil (2015-2024)
## Projeto G2 - Tema 13

---

## 1. Introducao

O comercio eletronico cresceu de forma expressiva no Brasil na ultima decada.
Este projeto analisa um dataset simulado de **4.440 registros** cobrindo **2015-2024**.

**Perguntas orientadoras:**
- Quais categorias possuem maior faturamento e lucro?
- Quais estados e regioes concentram mais vendas?
- Existem periodos sazonais de maior consumo?
- Como o faturamento evoluiu ao longo dos anos?
- Quais canais possuem melhor desempenho?

## 2. Importacao de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (12, 5)})
print('Bibliotecas carregadas com sucesso!')

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/Colab Notebooks/projeto-ecommerce/imagens', exist_ok=True)
print('Pasta imagens pronta!')

## 3. Leitura dos Dados

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/projeto-ecommerce/dados/simulacao_ecommerce_brasil.csv')
df['data'] = pd.to_datetime(df['data'])
print(f'Shape: {df.shape}')
df.head()

## 4. Limpeza e Preparacao dos Dados

In [ ]:
print('=== Valores nulos ===')
print(df.isnull().sum())
print(f'\nDuplicatas: {df.duplicated().sum()}')
print(f'\nFaturamentos negativos: {(df["faturamento"] < 0).sum()}')

In [ ]:
df.describe().round(2)

## 5. Engenharia de Atributos

In [ ]:
df['margem_lucro'] = (df['lucro'] / df['faturamento'] * 100).round(2)
df['semestre']     = df['mes'].apply(lambda m: '1 Sem' if m <= 6 else '2 Sem')
df['trimestre']    = pd.cut(df['mes'], bins=[0,3,6,9,12], labels=['T1','T2','T3','T4'])
df[['margem_lucro','semestre','trimestre']].head()

## 6. Analise Exploratoria

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['faturamento'], bins=40, color='#4a6cf7', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribuicao do Faturamento', fontweight='bold')
axes[0].set_xlabel('Faturamento (R$)')
axes[1].hist(df['lucro'], bins=40, color='#26c6da', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribuicao do Lucro', fontweight='bold')
axes[1].set_xlabel('Lucro (R$)')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Colab Notebooks/projeto-ecommerce/imagens/distribuicao.png', bbox_inches='tight')
plt.show()

In [ ]:
num_cols = ['quantidade','preco_unitario','faturamento','custo','lucro',
            'prazo_entrega','avaliacao_cliente','margem_lucro']
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, ax=ax, linewidths=0.5)
ax.set_title('Matriz de Correlacao', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Colab Notebooks/projeto-ecommerce/imagens/correlacao.png', bbox_inches='tight')
plt.show()

## 7. KPIs - Indicadores-Chave de Desempenho

In [ ]:
fat_total   = df['faturamento'].sum()
lucro_total = df['lucro'].sum()
kpis = {
    'Faturamento Total':         f'R$ {fat_total:,.0f}',
    'Lucro Total':               f'R$ {lucro_total:,.0f}',
    'Margem de Lucro':           f'{lucro_total/fat_total*100:.1f}%',
    'Ticket Medio':              f'R$ {df["faturamento"].mean():,.0f}',
    'Produto mais vendido':      df.groupby('produto')['quantidade'].sum().idxmax(),
    'Categoria mais lucrativa':  df.groupby('categoria')['lucro'].sum().idxmax(),
    'Regiao destaque':           df.groupby('regiao')['faturamento'].sum().idxmax(),
    'Prazo medio (dias)':        f'{df["prazo_entrega"].mean():.1f}',
    'Avaliacao media':           f'{df["avaliacao_cliente"].mean():.2f}/5.0',
}
print('='*55)
print('     INDICADORES-CHAVE DE DESEMPENHO')
print('='*55)
for k, v in kpis.items():
    print(f'  {k:<40} {v}')
print('='*55)

## 8. Visualizacoes

In [ ]:
# Evolucao anual do faturamento
fat_ano = df.groupby('ano')['faturamento'].sum()
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(fat_ano.index, fat_ano.values/1e6, marker='o', linewidth=2.5, color='#4a6cf7')
ax.fill_between(fat_ano.index, fat_ano.values/1e6, alpha=0.1, color='#4a6cf7')
ax.set_title('Evolucao Anual do Faturamento (R$ Milhoes)', fontsize=14, fontweight='bold')
ax.set_xlabel('Ano'); ax.set_ylabel('R$ Milhoes')
ax.grid(axis='y', alpha=0.4); sns.despine(ax=ax)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Colab Notebooks/projeto-ecommerce/imagens/evolucao_anual.png', bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap sazonalidade
meses_nomes = {1:'Jan',2:'Fev',3:'Mar',4:'Abr',5:'Mai',6:'Jun',
               7:'Jul',8:'Ago',9:'Set',10:'Out',11:'Nov',12:'Dez'}
pivot = df.pivot_table(values='faturamento', index='ano', columns='mes', aggfunc='sum')
pivot.columns = [meses_nomes[c] for c in pivot.columns]
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot/1e6, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label':'R$ Milhoes'})
ax.set_title('Heatmap de Sazonalidade - Faturamento por Mes e Ano', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Colab Notebooks/projeto-ecommerce/imagens/heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Faturamento e lucro por categoria
cat_df = df.groupby('categoria')[['faturamento','lucro']].sum().sort_values('faturamento')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cat_df['faturamento'].plot(kind='barh', ax=axes[0], color=sns.color_palette('Set2',6))
axes[0].set_title('Faturamento por Categoria', fontsize=13, fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R${x/1e6:.1f}M'))
axes[0].grid(axis='x', alpha=0.3); sns.despine(ax=axes[0])
cat_df['lucro'].plot(kind='barh', ax=axes[1], color=sns.color_palette('husl',6))
axes[1].set_title('Lucro por Categoria', fontsize=13, fontweight='bold')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R${x/1e6:.1f}M'))
axes[1].grid(axis='x', alpha=0.3); sns.despine(ax=axes[1])
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Colab Notebooks/projeto-ecommerce/imagens/cat_fat_lucro.png', bbox_inches='tight')
plt.show()

In [ ]:
# Top 10 estados por faturamento
uf_fat = df.groupby('uf')['faturamento'].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(12, 5))
uf_fat.plot(kind='bar', ax=ax, color=sns.color_palette('Blues_d', 10))
ax.set_title('Top 10 Estados por Faturamento', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R${x/1e6:.1f}M'))
ax.tick_params(axis='x', rotation=0); ax.grid(axis='y', alpha=0.3)
sns.despine(ax=ax)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Colab Notebooks/projeto-ecommerce/imagens/top10_uf.png', bbox_inches='tight')
plt.show()

In [ ]:
# Dispersao lucro x faturamento
fig, ax = plt.subplots(figsize=(10, 6))
palette = sns.color_palette('Set2', df['categoria'].nunique())
for i, cat in enumerate(df['categoria'].unique()):
    sub = df[df['categoria']==cat]
    ax.scatter(sub['faturamento'], sub['lucro'], label=cat, color=palette[i], alpha=0.5, s=30)
z = np.polyfit(df['faturamento'], df['lucro'], 1)
p = np.poly1d(z)
xs = np.linspace(df['faturamento'].min(), df['faturamento'].max(), 100)
ax.plot(xs, p(xs), 'k--', linewidth=1.2, label='Tendencia')
ax.set_title('Dispersao: Lucro x Faturamento por Categoria', fontsize=14, fontweight='bold')
ax.set_xlabel('Faturamento (R$)'); ax.set_ylabel('Lucro (R$)')
ax.legend(title='Categoria'); ax.grid(alpha=0.3); sns.despine(ax=ax)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Colab Notebooks/projeto-ecommerce/imagens/dispersao.png', bbox_inches='tight')
plt.show()

In [ ]:
# Boxplot margem de lucro por categoria
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df, x='categoria', y='margem_lucro', palette='Set2', ax=ax)
ax.set_title('Margem de Lucro por Categoria (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Categoria'); ax.set_ylabel('Margem (%)')
ax.tick_params(axis='x', rotation=15); ax.grid(axis='y', alpha=0.3); sns.despine(ax=ax)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Colab Notebooks/projeto-ecommerce/imagens/boxplot_margem.png', bbox_inches='tight')
plt.show()

In [ ]:
# Canal de venda - faturamento e avaliacao
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
canal_fat = df.groupby('canal_venda')['faturamento'].sum().sort_values(ascending=False)
canal_fat.plot(kind='bar', ax=axes[0], color=['#4a6cf7','#ff7043','#26c6da'])
axes[0].set_title('Faturamento por Canal de Venda', fontsize=13, fontweight='bold')
axes[0].set_ylabel('R$'); axes[0].tick_params(axis='x', rotation=0)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R${x/1e6:.1f}M'))
axes[0].grid(axis='y', alpha=0.3); sns.despine(ax=axes[0])
canal_aval = df.groupby('canal_venda')['avaliacao_cliente'].mean().sort_values(ascending=False)
canal_aval.plot(kind='bar', ax=axes[1], color=['#26c6da','#4a6cf7','#ff7043'])
axes[1].set_title('Avaliacao Media por Canal', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Nota (0-5)'); axes[1].set_ylim(0, 5)
axes[1].tick_params(axis='x', rotation=0); axes[1].grid(axis='y', alpha=0.3)
sns.despine(ax=axes[1])
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Colab Notebooks/projeto-ecommerce/imagens/canal_venda.png', bbox_inches='tight')
plt.show()

## 9. Interpretacao dos Resultados

### Evolucao Temporal
- O faturamento cresceu consistentemente ao longo dos 10 anos, refletindo a expansao do e-commerce.
- O heatmap evidencia picos em **novembro e dezembro** — Black Friday e festas de fim de ano.

### Analise Regional
- Sudeste e Sul concentram o maior volume; outras regioes apresentam ticket medio competitivo.

### Categorias
- Eletronicos e Moda lideram em faturamento; Beleza destaca-se em margem de lucro.
- A dispersao Lucro x Faturamento confirma correlacao positiva forte.

### Logistica e Satisfacao
- Prazo medio de entrega varia por regiao — indicador para acoes de melhoria operacional.
- Avaliacao media e consistente entre os canais de venda.

## 10. Conclusao

O e-commerce brasileiro demonstra trajetoria de crescimento robusta em 2015-2024.

**Principais insights:**
1. **Crescimento sustentado**: faturamento aumenta a cada ano.
2. **Sazonalidade clara**: novembro e dezembro sao os meses de maior consumo.
3. **Categorias estrategicas**: Eletronicos e Moda dominam em volume; Beleza se destaca em margem.
4. **Concentracao regional**: Sudeste e Sul lideram, com potencial em outras regioes.
5. **Canal digital em expansao**: aplicativo cresce em relevancia como canal de venda.

---
*Projeto desenvolvido para a disciplina de Analise e Visualizacao de Dados com Python.*